In [ ]:
# jax ecosystem
import jax

# jax.config.update("jax_enable_x64", False)
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "gpu")
jax.config.update("jax_debug_nans", False)

device = jax.local_devices()[0]
print(device.device_kind)

stats = device.memory_stats()
free = stats["bytes_limit"] - stats["bytes_in_use"]
print(f"Total VRAM : {stats['bytes_limit'] / 1024**3:.2f} GB")
print(f"In use     : {stats['bytes_in_use'] / 1024**3:.2f} GB")
print(f"Free       : {free / 1024**3:.2f} GB")

import jax.numpy as np
import jax.tree as jtu
import jax.random as jr

# amigo
import amigo
import dorito
import zodiax

# matplotlib ecosystem
import matplotlib.pyplot as plt
import matplotlib as mpl
import ehtplot
import scienceplots

# other
import pandas as pd
import os
import sys
# matplotlib parameters
plt.style.use(["science", "bright", "no-latex"])
plt.rcParams["image.cmap"] = "inferno"
plt.rcParams["font.family"] = "serif"
plt.rcParams["image.origin"] = "lower"
plt.rcParams["figure.dpi"] = 300
plt.rcParams["font.size"] = 8
plt.rcParams["xtick.direction"] = "out"
plt.rcParams["ytick.direction"] = "out"

inferno = mpl.colormaps["inferno"]
seismic = mpl.colormaps["seismic"]
coolwarm = mpl.colormaps["coolwarm"]
viridis = mpl.colormaps["viridis"]
plasma = mpl.colormaps["plasma"]
magma = mpl.colormaps["magma"]

inferno.set_bad("k", 0.5)
seismic.set_bad("k", 0.5)
coolwarm.set_bad("k", 0.5)
viridis.set_bad("k", 0.5)
plasma.set_bad("k", 0.5)
magma.set_bad("k", 0.5)

if jax.config.read("jax_enable_x64"):
    print("64bit enabled")
else:
    print("32bit enabled")

import equinox
import lineax

print("jax version:", jax.__version__)
print("zodiax version:", zodiax.__version__)
print("amigo version:", amigo.__version__)
print("dorito version:", dorito.__version__)

print("equinox version:", equinox.__version__)
print("lineax version", lineax.__version__)

In [ ]:
# Setting data path
from socket import gethostname

print(f"host name: {gethostname()}")

if (
    gethostname() == "maxs-mbp-14.shared.sydney.edu.au"
    or gethostname() == "Maxs-MacBook-Pro-14.local"
):
    data_dir = "/Volumes/morgana2/snert/max/data/JWST/"
    cache_dir = "/Volumes/morgana2/snert/max/data/amigo_cache"
    amigo_files_path = "/Volumes/morgana2/snert/max/data/amigo_files/v_0.0.10"
    output_path = "/Users/mc/outputs/retrain"

elif gethostname().startswith("max-"):
    data_dir = "/home/dgxuser/max/data/JWST/"
    cache_dir = "/home/dgxuser/max/data/amigo_cache"
    amigo_files_path = "/home/dgxuser/max/data/amigo_files/v_0.0.10"
    output_path = "/home/dgxuser/max/outputs/retrain"

else:
    data_dir = "/fred/oz440/max/data/JWST/"
    cache_dir = "/fred/oz440/max/data/amigo_cache"
    amigo_files_path = "/fred/oz440/max/data/amigo_files/v_0.0.10"
    output_path = "/fred/oz440/max/outputs/retrain"


print(f"amigo_files_path: {amigo_files_path}")

In [ ]:
# Add extra bad pixels
badpix_bool = np.load(f"{cache_dir}/full_badpix.npy")
badpix_bool = badpix_bool.at[25, 74].set(True)
badpix = np.array(badpix_bool, dtype=int)

def add_badpix(file):
    file["BADPIX"].data = badpix
    file["BADPIX"].data[25, 74] = 1
    return file


# Visualising metadata in a table
def summarise_files(files):
    prog_ids = []
    fnames = []
    targets = []
    filts = []
    diths = []
    ngroups = []
    time = []
    pis = []
    cals = []

    for file in files:
        header = file[0].header

        prog_ids.append(header["PROGRAM"][1:])
        fnames.append(header["FILENAME"][:25])
        targets.append(header["TARGPROP"])
        filts.append(header["FILTER"])
        diths.append(f"{header["PATT_NUM"]}/{header["NUMDTHPT"]}")
        ngroups.append(f"{header["NGROUPS"]}/{header["NINTS"]}")
        time.append(header["DATE-BEG"])
        pis.append(header["PI_NAME"])
        try:
            cals.append(header["IS_PSF"])
        except KeyError:
            cals.append("FLAT")

    df = pd.DataFrame(
        {
            "program": prog_ids,
            # "filename": fnames,
            "target": targets,
            "filter": filts,
            "dither": diths,
            "g/i": ngroups,
            "date": time,
            "PI": pis,
            "CAL": cals,
        }
    )
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)
    df = df.assign(date=pd.to_datetime(df["date"]).dt.strftime("%d-%m-%Y"))

    with pd.option_context(
        "display.expand_frame_repr",
        False,
        "display.max_columns",
        None,
        "display.max_rows",
        None,
        "display.width",
        1000,
    ):
        print(df)

    # df.to_excel("cal_data.xlsx", index=False)

In [ ]:
# LOADING IN DATA
from astropy.io import fits
from amigo.files import get_files
from dorito.misc import truncate_files


dark_files = []
print()
print("DARKS")
program_path = os.path.join(data_dir, "DARKS/uncal/")
files = get_files(program_path, "_uncal", EXP_TYPE="NIS_DARK")
dark_files += files

# dark_files = [add_badpix(file) for file in dark_files]
# truncate_files(dark_files, 500)  # TRUNCATING RAMP
summarise_files(dark_files)

file_800, file_050 = dark_files

In [ ]:
def line_fit(pix_ramp):
    """
    Fit a line to a single pixel dark ramp using unweighted least squares.

    Since this is dark data, noise is read-noise dominated and constant
    across all groups, so uniform weighting is appropriate.

    Parameters
    ----------
    pix_ramp : (N,) array
        Raw ramp samples for a single pixel, in electrons.

    Returns
    -------
    coeffs : (2,) array
        [slope, intercept] of the fitted line, where slope is the
        dark current in electrons per group.
    """

    # Group indices as x-axis
    x = np.arange(len(pix_ramp), dtype=float)

    # Design matrix: [x, 1] for slope and intercept
    A = np.column_stack([x, np.ones_like(x)])

    # Unweighted least squares — valid since read noise is uniform across groups
    coeffs, _, _, _ = np.linalg.lstsq(A, pix_ramp, rcond=None)

    return coeffs


In [ ]:
from jax import vmap, jit

ramp = np.array(file_050["SCI"].data)  # (nint, ngroup, npix, npix)
print(ramp.shape)

# moving axes around to make vmap work
ramp_t = np.moveaxis(ramp, (1), (-1))  # (nint, npix, npix, ngroup)
print(ramp_t.shape)

# vmapping over pixels and integrations
fit_all_pix = jit(vmap(vmap(vmap(line_fit))))

# fitting all pixels
coeffs = fit_all_pix(ramp_t)  # (nints, npix, npix, n_coeffs)

dark_currents = coeffs[..., 0]  # (80, 80)
biases = coeffs[..., 1]  # (80, 80)

mean_dc = dark_currents.mean(0).at[badpix_bool].set(np.nan)
std_dc = dark_currents.std(0).at[badpix_bool].set(np.nan)

mean_bias = biases.mean(0).at[badpix_bool].set(np.nan)
std_bias = biases.std(0).at[badpix_bool].set(np.nan)

In [ ]:
fig, ax = plt.subplots(4, 3, figsize=(11, 11))

# DARK CURRENT
mean = np.nanmedian(mean_dc)
im = ax[0][0].imshow(mean_dc, plasma)
ax[0][0].set(title=f"Dark Current: Median {mean:.2f}")
ax[0][0].axis("off")
fig.colorbar(im, ax=ax[0][0])
ax[1][0].hist(mean_dc.ravel(), bins=50)
ax[1][0].set(xlabel="Counts per group")

# ERROR ON DARK CURRENT
mean = np.nanmedian(std_dc)
im = ax[0][1].imshow(std_dc, viridis)
ax[0][1].set(title=f"Error on Dark Current: Median {mean:.2f}")
ax[0][1].axis("off")
fig.colorbar(im, ax=ax[0][1])
ax[1][1].hist(std_dc.ravel(), bins=50, log=True)
ax[1][1].set(xlabel="Error on counts per group")

# SNR OF DARK CURRENT
mean = np.nanmedian(mean_dc / std_dc)
im = ax[0][2].imshow(mean_dc / std_dc, inferno)
ax[0][2].set(title=f"SNR of Dark Current: Median {mean:.2f}")
ax[0][2].axis("off")
fig.colorbar(im, ax=ax[0][2])
ax[1][2].hist((mean_dc / std_dc).ravel(), bins=50, log=False)
ax[1][2].set(xlabel="Error on counts per group")

# BIASES
mean = np.nanmedian(mean_bias)
im = ax[2][0].imshow(mean_bias, magma)
ax[2][0].set(title=f"Bias: Median {mean:.2f}")
ax[2][0].axis("off")
fig.colorbar(im, ax=ax[2][0])
ax[3][0].hist(mean_bias.ravel(), bins=50, log=False)
ax[3][0].set(xlabel="Counts", xlim=(0, None))

# ERROR ON BIAS
mean = np.nanmedian(std_bias)
im = ax[2][1].imshow(std_bias, viridis)
ax[2][1].set(title=f"Error on Biases: Median {mean:.2f}")
ax[2][1].axis("off")
fig.colorbar(im, ax=ax[2][1])
ax[3][1].hist(std_bias.ravel(), bins=50, log=True)
ax[3][1].set(xlabel="Error on counts")

# SNR OF BIAS
mean = np.nanmedian(mean_bias / std_bias)
im = ax[2][2].imshow(mean_bias / std_bias, inferno)
ax[2][2].set(title=f"SNR of Bias: Median {mean:.2f}")
ax[2][2].axis("off")
fig.colorbar(im, ax=ax[2][2])
ax[3][2].hist((mean_bias / std_bias).ravel(), bins=50, log=False)
ax[3][2].set(xlabel="Error on counts per group")

plt.tight_layout()
plt.savefig("dark_analysis.png", dpi=400)
plt.show()